In [11]:
import colorama
from colorama import Fore, Style
from IPython.display import display

def preview(file, sheet_name, n=5):
    print(Fore.BLUE+Style.BRIGHT+f"\n* Preview of '{sheet_name}' ({file.split('/')[-1]}) *\n")
    
    df = pd.read_excel(file, sheet_name=sheet_name)

    # Select numeric columns
    numeric_cols = df.select_dtypes(include="number").columns

    if len(numeric_cols) > 0:
        # Keep only rows that contain at least one numeric value (not all NaN)
        df = df[df[numeric_cols].notna().any(axis=1)]

    # Show first n rows
    df_styled = df.head(n).style.set_properties(**{'text-align': 'left'})
    df_styled = df_styled.set_table_styles(
        [{'selector': 'th', 'props': [('text-align', 'left')]}]
    )

    display(df_styled)


In [13]:
import re
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils.cell import coordinate_from_string

# Specify the path to the Excel file
input_file = '/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/clean file to learn.xlsx'
output_mapping_file = '/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/final_formula_mapping_semantic.xlsx'
id_columns = ['Variable', 'Sector', 'Unit', 'Unnamed: 3']
year_col_map = {5 + i: year for i, year in enumerate(range(2025, 2055, 5))}  # columns E to J

# Load the structured ESM output produced by Script 02
wb = load_workbook(input_file, data_only=False)
ws_ind = wb["IND"]
ind_df = pd.read_excel(input_file, sheet_name="IND")
energy_df = pd.read_excel(input_file, sheet_name="Energy input per carrier")
tech_df = pd.read_excel(input_file, sheet_name="Tech, Carrier & Sector by Year")
heat_df = pd.read_excel(input_file, sheet_name="Electricity & Heat")

#Extract sheet names and cell references from a formula.
def extract_all_formula_references(formula_string):
    cleaned = formula_string.strip().lstrip("=")
    cleaned = cleaned.replace("'", "")
    pattern = r'([A-Za-z0-9 &,\-\']+?)!([A-Z]{1,3}[0-9]{1,5})'
    return re.findall(pattern, cleaned)

# Build semantic mapping
mapping = []

for row_idx in range(2, ws_ind.max_row + 1):
    try:
        ind_row = ind_df.iloc[row_idx - 2]
        row_id = {col: ind_row[col] for col in id_columns}
    except:
        continue

    for col_idx in range(5, 11):  # E to J → 2025 to 2050
        formula = ws_ind.cell(row=row_idx, column=col_idx).value
        if not isinstance(formula, str) or not formula.startswith("="):
            continue

        year = year_col_map.get(col_idx)
        multiplier = -1000 if "*-1000" in formula else 1000 if "*1000" in formula else 1

        refs = extract_all_formula_references(formula)
        for sheet_name, cell_ref in refs:
            try:
                col_letter, ref_row = coordinate_from_string(cell_ref)
                ref_row = int(ref_row)
            except:
                continue

            sheet_name = sheet_name.strip()

            try:
                if sheet_name == "Energy input per carrier":
                    row_data = energy_df.iloc[ref_row - 2]
                    mapping.append({
                        **row_id,
                        "year": year,
                        "source_type": "energy_input",
                        "carrier": row_data["Energy Carrier"],
                        "sector": row_data["Sector"],
                        "technology": "",
                        "source_block": "",
                        "multiplier": multiplier
                    })

                elif sheet_name == "Tech, Carrier & Sector by Year":
                    row_data = tech_df.iloc[ref_row - 2]
                    mapping.append({
                        **row_id,
                        "year": year,
                        "source_type": "tech_carrier",
                        "technology": row_data["Technology"],
                        "carrier": row_data["Energy Carrier"],
                        "sector": row_data["Sector"],
                        "source_block": "",
                        "multiplier": multiplier
                    })

                elif sheet_name == "Electricity & Heat":
                    row_data = heat_df.iloc[ref_row - 2]
                    # Infer whether it's electricity or heat by looking at previous "header" row
                    source_block = ""
                    for i in range(ref_row - 2, -1, -1):
                        if pd.isna(heat_df.iloc[i]["Technology"]) and pd.isna(heat_df.iloc[i]["Energy Carrier"]) is False:
                            source_block = str(heat_df.iloc[i]["Energy Carrier"]).strip()
                            break

                    mapping.append({
                        **row_id,
                        "year": year,
                        "source_type": "electricity_heat",
                        "technology": row_data["Technology"],
                        "carrier": row_data["Energy Carrier"],
                        "sector": "",  # not defined in that sheet
                        "source_block": "",
                        "multiplier": multiplier
                    })
            except:
                continue

# Save mapping
mapping_df = pd.DataFrame(mapping).drop_duplicates()
mapping_df.to_excel(output_mapping_file, index=False)
print(f"✔ Mapping saved to: {output_mapping_file}")


✔ Mapping saved to: /Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/final_formula_mapping_semantic.xlsx


In [17]:

mapping_file = '/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/final_formula_mapping_semantic.xlsx'
source_file = '/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/Book-EnergyBal with biofuels.xlsx'

# sheet where the rebuilt IND is written
sheet_name_to_add = 'IND_rebuilt'  # Sheet name for output

#Load mapping and source data
mapping_df = pd.read_excel(mapping_file)
energy_df = pd.read_excel(source_file, sheet_name="Energy input per carrier")
tech_df = pd.read_excel(source_file, sheet_name="Tech, Carrier & Sector by Year")
heat_df = pd.read_excel(source_file, sheet_name="Electricity & Heat")


id_cols = ['Variable', 'Sector', 'Unit', 'Unnamed: 3']
all_years = sorted(mapping_df['year'].unique())
reconstructed = []

# Recalculate based on mapping 
for key, group in mapping_df.groupby(id_cols):
    row = dict(zip(id_cols, key))

    for year in all_years:
        year_val = 0
        year_group = group[group["year"] == year]

        for _, entry in year_group.iterrows():
            try:
                val = 0

                if entry["source_type"] == "energy_input":
                    source_row = energy_df[
                        (energy_df["Energy Carrier"] == entry["carrier"]) &
                        (energy_df["Sector"] == entry["sector"])
                    ]
                    if not source_row.empty:
                        val = source_row.iloc[0].get(str(year), 0)

                elif entry["source_type"] == "tech_carrier":
                    source_row = tech_df[
                        (tech_df["Technology"] == entry["technology"]) &
                        (tech_df["Energy Carrier"] == entry["carrier"]) &
                        (tech_df["Sector"] == entry["sector"])
                    ]
                    if not source_row.empty:
                        val = source_row.iloc[0].get(str(year), 0)

                elif entry["source_type"] == "electricity_heat":
                    source_row = heat_df[
                        (heat_df["Technology"] == entry["technology"]) &
                        (heat_df["Energy Carrier"] == entry["carrier"])
                    ]
                    if not source_row.empty:
                        val = source_row.iloc[0].get(str(year), 0)

                # Safely handle NaN
                if pd.isna(val):
                    val = 0

                year_val += val * entry["multiplier"]

            except Exception as e:
                continue  # Silent fail for bad entries

        row[str(year)] = year_val

    reconstructed.append(row)

# Convert to DataFrame
reconstructed_df = pd.DataFrame(reconstructed)
reconstructed_df = reconstructed_df[id_cols + [str(y) for y in all_years]]

# Save to existing Excel as new sheet
with pd.ExcelWriter(source_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    reconstructed_df.to_excel(writer, sheet_name=sheet_name_to_add, index=False)

print(f"✔ Rebuilt IND sheet saved in '{sheet_name_to_add}' of: {source_file}")


✔ Rebuilt IND sheet saved in 'IND_rebuilt' of: /Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/Book-EnergyBal with biofuels.xlsx


In [18]:
preview(source_file, "IND_rebuilt")


* Preview of 'IND_rebuilt' (Book-EnergyBal with biofuels.xlsx) *



,Variable,Sector,Unit,Unnamed: 3,2025,2030,2035,2040,2045,2050
0,Electricity,Biomass,TJ,PGT05,8936.662724,8936.656524,397.877582,6697.948035,4631.646234,3830.016548
1,Electricity,CCS Bio,TJ,PGT12,0.003054,0.006609,4062.624813,2.492120,522.092016,349.413534
2,Electricity,CCS Gas,TJ,PGT11,0.050206,0.108225,0.218573,0.001395,0.028436,0.112519
3,Electricity,Gas fired,TJ,PGT03,108023.484996,123917.391606,58413.773743,40803.174787,40364.639408,41141.019498
4,Electricity,Hydro electric,TJ,PGT06,0.007915,0.118404,0.014265,0.000000,0.000000,0.032099
